In [1]:
# ============================================================
# 公共代码框 1：
# 读取 5 变量 + 构造多变量 HR/LR + 标准化 + 测试集 memmap
# ============================================================
# 输入 testx : (N, 58, 94, 10)
# 输出 testy : (N, 116, 188, 25)
#
# 变量顺序：
# VAR_NAMES = ["slp", "z300", "z500", "u10", "v10"]
#
# testx 通道：
# slp_t, slp_t4, z300_t, z300_t4, z500_t, z500_t4, u10_t, u10_t4, v10_t, v10_t4
#
# testy 通道：
# slp_t..slp_t4, z300_t..z300_t4, z500_t..z500_t4, u10_t..u10_t4, v10_t..v10_t4
# ============================================================

import os
import gc
import json
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm
from numpy.lib.format import open_memmap

warnings.filterwarnings("ignore")

# ============================================================
# 1. 路径与基本参数
# ============================================================
DATA_DIR = Path(r"H:\ERA5-6hour")
MODEL_DIR = Path(r"E:\Dr_Research\model")
RESULT_DIR = Path(r"E:\Dr_Research\result")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TMP_DIR = RESULT_DIR / "_tmp_downscaling_5vars_data"
TMP_DIR.mkdir(parents=True, exist_ok=True)

TEST_SIZE = 0.2

TIME_START = "1980-01-01T00:00:00"
TIME_END   = "2014-12-31T18:00:00"

LAT_RANGE = (-5.0, 53.0)
LON_RANGE = (93.0, 187.0)

VAR_NAMES = ["slp", "z300", "z500", "u10", "v10"]
FEATURE_HOURS = np.array([0, 6, 12, 18, 24], dtype=np.int32)

ALL_FEATURE_INDEX = np.arange(25, dtype=np.int32)
SPATIAL_ONLY_FEATURE_INDEX = np.array(
    [0, 4, 5, 9, 10, 14, 15, 19, 20, 24],
    dtype=np.int32
)
MID_FEATURE_INDEX = np.array(
    [i for i in range(25) if i not in set(SPATIAL_ONLY_FEATURE_INDEX.tolist())],
    dtype=np.int32
)

TESTX_NPY = TMP_DIR / "testx_data_standardized.npy"
TESTY_NPY = TMP_DIR / "testy_data_standardized.npy"
META_NPZ  = TMP_DIR / "metadata_data_standardized.npz"

# 重新构造数据时设为 True；已经构造过、只想重新算模型或指标时，可改为 False
REBUILD_DATA = True

VARIABLE_CONFIGS = {
    "slp": {
        "file": "Mean-sea-level-pressure-1980-2024.nc",
        "nc_var_candidates": ["msl", "psl", "slp", "mean_sea_level_pressure"],
        "level": None,
    },
    "z300": {
        "file": "Geopotential-300hpa-1980-2024.nc",
        "nc_var_candidates": ["z", "z300", "g300", "geopotential"],
        "level": 300,
    },
    "z500": {
        "file": "Geopotential-500hpa-1980-2024.nc",
        "nc_var_candidates": ["z", "z500", "g500", "geopotential"],
        "level": 500,
    },
    "u10": {
        "file": "10m-u-component-of-wind-1980-2024.nc",
        "nc_var_candidates": ["u10", "u", "10u", "u_component_of_wind_10m"],
        "level": None,
    },
    "v10": {
        "file": "10m-v-component-of-wind-1980-2024.nc",
        "nc_var_candidates": ["v10", "v", "10v", "v_component_of_wind_10m"],
        "level": None,
    },
}


# ============================================================
# 2. 读取与预处理函数
# ============================================================
def _find_dim_name(da, dim_keywords):
    for d in da.dims:
        dl = d.lower()
        if any(k in dl for k in dim_keywords):
            return d
    raise ValueError(f"无法在 dims={da.dims} 中找到维度关键词: {dim_keywords}")


def _guess_var_name(ds, candidates):
    for v in candidates:
        if v in ds.data_vars:
            return v

    valid_vars = []
    for v in ds.data_vars:
        if ds[v].ndim >= 3:
            valid_vars.append(v)

    if len(valid_vars) == 1:
        return valid_vars[0]

    raise ValueError(
        f"无法自动识别变量名。候选={candidates}, 文件内变量={list(ds.data_vars)}"
    )


def _select_level_if_needed(da, level_value):
    if level_value is None:
        return da

    level_dim_candidates = [
        d for d in da.dims
        if d.lower() in ["level", "pressure_level", "isobaricinhpa", "plev"]
        or "level" in d.lower()
        or "pressure" in d.lower()
    ]

    if len(level_dim_candidates) == 0:
        return da

    level_dim = level_dim_candidates[0]
    coord_values = da[level_dim].values

    if len(coord_values) == 1:
        return da.isel({level_dim: 0})

    return da.sel({level_dim: level_value}, method="nearest")


def _select_lat_lon(da, lat_range, lon_range):
    lat_dim = _find_dim_name(da, ["lat", "latitude"])
    lon_dim = _find_dim_name(da, ["lon", "longitude"])

    lat_values = da[lat_dim].values
    lon_values = da[lon_dim].values

    lat_min, lat_max = lat_range
    lon_min, lon_max = lon_range

    if lat_values[0] < lat_values[-1]:
        da = da.sel({lat_dim: slice(lat_min, lat_max)})
    else:
        da = da.sel({lat_dim: slice(lat_max, lat_min)})

    lon_values = da[lon_dim].values

    if lon_values.min() >= 0 and lon_values.max() > 180:
        da = da.sel({lon_dim: slice(lon_min, lon_max)})
    else:
        lon_min_180 = ((lon_min + 180) % 360) - 180
        lon_max_180 = ((lon_max + 180) % 360) - 180

        if lon_min_180 <= lon_max_180:
            da = da.sel({lon_dim: slice(lon_min_180, lon_max_180)})
        else:
            part1 = da.sel({lon_dim: slice(lon_min_180, 180)})
            part2 = da.sel({lon_dim: slice(-180, lon_max_180)})
            da = xr.concat([part1, part2], dim=lon_dim)

    return da


def _select_time_range(da, time_start, time_end):
    time_dim = _find_dim_name(da, ["time"])
    da = da.sortby(time_dim)

    try:
        da = da.sel({time_dim: slice(np.datetime64(time_start), np.datetime64(time_end))})
    except Exception:
        times = pd.to_datetime(da[time_dim].values)
        mask = (times >= pd.Timestamp(time_start)) & (times <= pd.Timestamp(time_end))
        da = da.isel({time_dim: np.where(mask)[0]})

    return da


def read_one_variable(cfg):
    nc_path = DATA_DIR / cfg["file"]
    if not nc_path.exists():
        raise FileNotFoundError(f"文件不存在: {nc_path}")

    ds = xr.open_dataset(nc_path)

    var_name = _guess_var_name(ds, cfg["nc_var_candidates"])
    da = ds[var_name]

    da = _select_level_if_needed(da, cfg["level"])
    da = _select_lat_lon(da, LAT_RANGE, LON_RANGE)
    da = _select_time_range(da, TIME_START, TIME_END)

    time_dim = _find_dim_name(da, ["time"])
    lat_dim = _find_dim_name(da, ["lat", "latitude"])
    lon_dim = _find_dim_name(da, ["lon", "longitude"])

    da = da.transpose(time_dim, lat_dim, lon_dim)

    times = pd.to_datetime(da[time_dim].values)
    lat = da[lat_dim].values
    lon = da[lon_dim].values

    arr = da.values.astype(np.float32)

    ds.close()
    del ds, da
    gc.collect()

    return arr, times, lat, lon, var_name


def build_hr_lr_samples_single_var(arr, times, lat, lon):
    """
    从原始 0.25° / 6-hour 数据构造：
    HR: 0.5° / 6-hour, 5 个时次
    LR: 1.0° / 1-day, 2 个时次

    复现旧 notebook：
        arr_05 = arr[:, ::2, ::2]
        HR = arr_05[:, :-1, :-1]
        LR = arr_05[:, :-1:2, :-1:2]
    """

    T = arr.shape[0]

    arr_05 = arr[:, ::2, ::2]
    lat_05 = lat[::2]
    lon_05 = lon[::2]

    arr_hr_base = arr_05[:, :-1, :-1]
    arr_lr_base = arr_05[:, :-1:2, :-1:2]

    lat_hr = lat_05[:-1]
    lon_hr = lon_05[:-1]
    lat_lr = lat_05[:-1:2]
    lon_lr = lon_05[:-1:2]

    candidate_start = np.arange(0, T - 4)
    n = len(candidate_start)

    hr = np.empty(
        (n, arr_hr_base.shape[1], arr_hr_base.shape[2], 5),
        dtype=np.float32
    )
    lr = np.empty(
        (n, arr_lr_base.shape[1], arr_lr_base.shape[2], 2),
        dtype=np.float32
    )

    hr[..., 0] = arr_hr_base[candidate_start,     :, :]
    hr[..., 1] = arr_hr_base[candidate_start + 1, :, :]
    hr[..., 2] = arr_hr_base[candidate_start + 2, :, :]
    hr[..., 3] = arr_hr_base[candidate_start + 3, :, :]
    hr[..., 4] = arr_hr_base[candidate_start + 4, :, :]

    lr[..., 0] = arr_lr_base[candidate_start,     :, :]
    lr[..., 1] = arr_lr_base[candidate_start + 4, :, :]

    sample_times = times[candidate_start]

    return hr, lr, sample_times, lat_hr, lon_hr, lat_lr, lon_lr


def standardize_like_old_notebook(hr, lr):
    hr_mean = np.nanmean(hr, axis=0).astype(np.float32)
    hr_std  = np.nanstd(hr, axis=0).astype(np.float32)

    lr_mean = np.nanmean(lr, axis=0).astype(np.float32)
    lr_std  = np.nanstd(lr, axis=0).astype(np.float32)

    hr_std = np.where((hr_std == 0) | np.isnan(hr_std), 1.0, hr_std).astype(np.float32)
    lr_std = np.where((lr_std == 0) | np.isnan(lr_std), 1.0, lr_std).astype(np.float32)

    hr_norm = ((hr - hr_mean) / hr_std).astype(np.float32)
    lr_norm = ((lr - lr_mean) / lr_std).astype(np.float32)

    hr_range_crop = np.array(
        [
            np.nanmax(hr_norm[:, 1:-1, 1:-1, k]) - np.nanmin(hr_norm[:, 1:-1, 1:-1, k])
            for k in range(5)
        ],
        dtype=np.float32
    )
    hr_range_crop = np.where(
        (hr_range_crop == 0) | np.isnan(hr_range_crop),
        1.0,
        hr_range_crop
    ).astype(np.float32)

    return hr_norm, lr_norm, hr_mean, hr_std, lr_mean, lr_std, hr_range_crop


# ============================================================
# 3. 构造多变量 testx/testy
# ============================================================
def build_or_load_multivar_test_data():
    if (not REBUILD_DATA) and TESTX_NPY.exists() and TESTY_NPY.exists() and META_NPZ.exists():
        print("读取已有 memmap 数据。")
        testx = np.load(TESTX_NPY, mmap_mode="r")
        testy = np.load(TESTY_NPY, mmap_mode="r")
        meta = np.load(META_NPZ, allow_pickle=True)

        metadata = {
            "test_times": meta["test_times"],
            "lat": meta["lat"],
            "lon": meta["lon"],
            "lat_lr": meta["lat_lr"],
            "lon_lr": meta["lon_lr"],
            "hr_range_crop": meta["hr_range_crop"],
            "split_index": int(meta["split_index"]),
        }

        print("testx:", testx.shape)
        print("testy:", testy.shape)
        return testx, testy, metadata

    for p in [TESTX_NPY, TESTY_NPY, META_NPZ]:
        if p.exists():
            p.unlink()

    testx_mm = None
    testy_mm = None

    hr_range_crop_all = np.zeros((25,), dtype=np.float32)
    test_times_final = None
    lat_hr_final = None
    lon_hr_final = None
    lat_lr_final = None
    lon_lr_final = None
    split_index_final = None

    for vi, var_name in enumerate(VAR_NAMES):
        cfg = VARIABLE_CONFIGS[var_name]
        print(f"\n========== 读取并处理变量: {var_name} ==========")

        raw, times, lat, lon, nc_var_name = read_one_variable(cfg)

        print(f"[{var_name}] nc变量名:", nc_var_name)
        print(f"[{var_name}] raw shape:", raw.shape)
        print(f"[{var_name}] time:", times[0], "->", times[-1])
        print(f"[{var_name}] lat size:", len(lat), "lon size:", len(lon))

        hr, lr, sample_times, lat_hr, lon_hr, lat_lr, lon_lr = build_hr_lr_samples_single_var(
            raw, times, lat, lon
        )

        del raw
        gc.collect()

        print(f"[{var_name}] HR before norm:", hr.shape)
        print(f"[{var_name}] LR before norm:", lr.shape)

        hr_norm, lr_norm, hr_mean, hr_std, lr_mean, lr_std, hr_range_crop = standardize_like_old_notebook(hr, lr)

        split_index = int((1.0 - TEST_SIZE) * hr_norm.shape[0])

        if testx_mm is None:
            n_test = hr_norm.shape[0] - split_index
            h_lr, w_lr = lr_norm.shape[1], lr_norm.shape[2]
            h_hr, w_hr = hr_norm.shape[1], hr_norm.shape[2]

            testx_mm = open_memmap(
                TESTX_NPY,
                mode="w+",
                dtype=np.float32,
                shape=(n_test, h_lr, w_lr, 10)
            )
            testy_mm = open_memmap(
                TESTY_NPY,
                mode="w+",
                dtype=np.float32,
                shape=(n_test, h_hr, w_hr, 25)
            )

            test_times_final = np.array(sample_times[split_index:], dtype="datetime64[ns]")
            lat_hr_final = lat_hr.astype(np.float32)
            lon_hr_final = lon_hr.astype(np.float32)
            lat_lr_final = lat_lr.astype(np.float32)
            lon_lr_final = lon_lr.astype(np.float32)
            split_index_final = split_index

        else:
            if split_index != split_index_final:
                raise ValueError(f"{var_name} 的 split_index 与前面变量不一致。")
            if not np.array_equal(np.array(sample_times[split_index:], dtype="datetime64[ns]"), test_times_final):
                raise ValueError(f"{var_name} 的测试集时间与前面变量不一致。")

        x0 = vi * 2
        y0 = vi * 5

        testx_mm[..., x0:x0+2] = lr_norm[split_index:].astype(np.float32)
        testy_mm[..., y0:y0+5] = hr_norm[split_index:].astype(np.float32)
        hr_range_crop_all[y0:y0+5] = hr_range_crop

        testx_mm.flush()
        testy_mm.flush()

        print(f"[{var_name}] 写入 testx 通道 {x0}:{x0+2}")
        print(f"[{var_name}] 写入 testy 通道 {y0}:{y0+5}")

        del hr, lr, hr_norm, lr_norm, hr_mean, hr_std, lr_mean, lr_std
        gc.collect()

    np.savez(
        META_NPZ,
        test_times=test_times_final,
        lat=lat_hr_final,
        lon=lon_hr_final,
        lat_lr=lat_lr_final,
        lon_lr=lon_lr_final,
        hr_range_crop=hr_range_crop_all,
        split_index=np.array(split_index_final, dtype=np.int64),
    )

    testx_mm.flush()
    testy_mm.flush()

    testx = np.load(TESTX_NPY, mmap_mode="r")
    testy = np.load(TESTY_NPY, mmap_mode="r")
    meta = np.load(META_NPZ, allow_pickle=True)

    metadata = {
        "test_times": meta["test_times"],
        "lat": meta["lat"],
        "lon": meta["lon"],
        "lat_lr": meta["lat_lr"],
        "lon_lr": meta["lon_lr"],
        "hr_range_crop": meta["hr_range_crop"],
        "split_index": int(meta["split_index"]),
    }

    print("\n========== 多变量数据构造完成 ==========")
    print("testx:", testx.shape)
    print("testy:", testy.shape)
    print("test time:", metadata["test_times"][0], "->", metadata["test_times"][-1])
    print("HR lat/lon:", len(metadata["lat"]), len(metadata["lon"]))
    print("LR lat/lon:", len(metadata["lat_lr"]), len(metadata["lon_lr"]))

    return testx, testy, metadata


testx, testy, metadata = build_or_load_multivar_test_data()

print("\nALL_FEATURE_INDEX:", ALL_FEATURE_INDEX.tolist())
print("SPATIAL_ONLY_FEATURE_INDEX:", SPATIAL_ONLY_FEATURE_INDEX.tolist())
print("MID_FEATURE_INDEX:", MID_FEATURE_INDEX.tolist())


# ============================================================
# 4. 指标计算函数：R, MSE, SSIM, PSNR
# ============================================================
def feature_to_var_and_hour(feature_index):
    vi = int(feature_index // 5)
    hi = int(feature_index % 5)
    return VAR_NAMES[vi], int(FEATURE_HOURS[hi])


def calc_feature_r_mse_spatial_chunk(y_true, y_pred, feature_index, lat_chunk=12):
    """
    y_true/y_pred shape: (N, H, W, 25)
    对单个 feature 计算：
    1. 每个格点沿 time 计算 Pearson R，然后空间平均
    2. 每个格点沿 time 计算 MSE，然后空间平均
    """

    n, h, w, _ = y_true.shape

    r_sum = 0.0
    r_count = 0
    mse_sum = 0.0
    mse_count = 0

    for i0 in range(1, h - 1, lat_chunk):
        i1 = min(h - 1, i0 + lat_chunk)

        yt = np.asarray(y_true[:, i0:i1, 1:-1, feature_index], dtype=np.float64)
        yp = np.asarray(y_pred[:, i0:i1, 1:-1, feature_index], dtype=np.float64)

        mse_map = np.nanmean((yp - yt) ** 2, axis=0)
        valid_mse = np.isfinite(mse_map)
        if np.any(valid_mse):
            mse_sum += float(np.nansum(mse_map[valid_mse]))
            mse_count += int(np.sum(valid_mse))

        yt_mean = np.nanmean(yt, axis=0, keepdims=True)
        yp_mean = np.nanmean(yp, axis=0, keepdims=True)

        yt_anom = yt - yt_mean
        yp_anom = yp - yp_mean

        numerator = np.nansum(yt_anom * yp_anom, axis=0)
        denominator = np.sqrt(
            np.nansum(yt_anom ** 2, axis=0) *
            np.nansum(yp_anom ** 2, axis=0)
        )

        r_map = numerator / denominator
        valid_r = np.isfinite(r_map)
        if np.any(valid_r):
            r_sum += float(np.nansum(r_map[valid_r]))
            r_count += int(np.sum(valid_r))

        del yt, yp, yt_mean, yp_mean, yt_anom, yp_anom, numerator, denominator, r_map, mse_map
        gc.collect()

    r = r_sum / r_count if r_count > 0 else np.nan
    mse = mse_sum / mse_count if mse_count > 0 else np.nan

    return float(r), float(mse)


def calc_feature_ssim_psnr_tf(y_true, y_pred, feature_index, max_val, batch_size=16):
    """
    使用 TensorFlow 分 batch 计算单 feature 的 SSIM/PSNR。
    默认 batch_size 很小，避免显存或内存峰值过大。
    """

    import tensorflow as tf

    n = y_true.shape[0]

    if (not np.isfinite(max_val)) or max_val <= 0:
        max_val = 1.0

    ssim_sum = 0.0
    psnr_sum = 0.0
    count = 0

    with tf.device("/CPU:0"):
        for b0 in range(0, n, batch_size):
            b1 = min(n, b0 + batch_size)

            yt = np.asarray(y_true[b0:b1, 1:-1, 1:-1, feature_index], dtype=np.float32)
            yp = np.asarray(y_pred[b0:b1, 1:-1, 1:-1, feature_index], dtype=np.float32)

            yt = np.nan_to_num(yt, nan=0.0, posinf=0.0, neginf=0.0)[..., np.newaxis]
            yp = np.nan_to_num(yp, nan=0.0, posinf=0.0, neginf=0.0)[..., np.newaxis]

            ssim_batch = tf.image.ssim(yp, yt, max_val=float(max_val)).numpy()
            psnr_batch = tf.image.psnr(yp, yt, max_val=float(max_val)).numpy()

            valid_ssim = np.isfinite(ssim_batch)
            valid_psnr = np.isfinite(psnr_batch)

            if np.any(valid_ssim):
                ssim_sum += float(np.nansum(ssim_batch[valid_ssim]))
            if np.any(valid_psnr):
                psnr_sum += float(np.nansum(psnr_batch[valid_psnr]))

            count += int(b1 - b0)

            del yt, yp, ssim_batch, psnr_batch
            gc.collect()

    ssim = ssim_sum / count if count > 0 else np.nan
    psnr = psnr_sum / count if count > 0 else np.nan

    return float(ssim), float(psnr)


def calc_feature_metrics_table(y_true, y_pred, hr_range_crop, model_name, ssim_batch_size=16):
    rows = []

    for k in tqdm(range(25), desc=f"Feature metrics: {model_name}"):
        var_name, hour = feature_to_var_and_hour(k)

        r, mse = calc_feature_r_mse_spatial_chunk(
            y_true=y_true,
            y_pred=y_pred,
            feature_index=k,
            lat_chunk=12
        )

        ssim, psnr = calc_feature_ssim_psnr_tf(
            y_true=y_true,
            y_pred=y_pred,
            feature_index=k,
            max_val=float(hr_range_crop[k]),
            batch_size=ssim_batch_size
        )

        rows.append({
            "Model": model_name,
            "Variable": var_name,
            "Feature_Index": k,
            "Hour": hour,
            "R": r,
            "MSE": mse,
            "SSIM": ssim,
            "PSNR": psnr,
        })

        gc.collect()

    return pd.DataFrame(rows)


def aggregate_metrics_from_feature_table(feature_table, model_name):
    metric_cols = ["R", "MSE", "SSIM", "PSNR"]

    rows_all = []
    rows_mid = []

    for var_name in VAR_NAMES:
        all_part = feature_table[feature_table["Variable"] == var_name]
        mid_part = all_part[~all_part["Feature_Index"].isin(SPATIAL_ONLY_FEATURE_INDEX.tolist())]

        rows_all.append({
            "Model": model_name,
            "Variable": var_name,
            **all_part[metric_cols].mean(numeric_only=True).to_dict()
        })

        rows_mid.append({
            "Model": model_name,
            "Variable": var_name,
            **mid_part[metric_cols].mean(numeric_only=True).to_dict()
        })

    table_all_by_variable = pd.DataFrame(rows_all)
    table_mid_by_variable = pd.DataFrame(rows_mid)

    table_all_overall = (
        table_all_by_variable
        .groupby("Model", as_index=False)[metric_cols]
        .mean()
    )

    table_mid_overall = (
        table_mid_by_variable
        .groupby("Model", as_index=False)[metric_cols]
        .mean()
    )

    return table_all_overall, table_all_by_variable, table_mid_overall, table_mid_by_variable


def save_metrics_tables(feature_tables, out_xlsx):
    metric_cols = ["R", "MSE", "SSIM", "PSNR"]

    all_by_variable_list = []
    mid_by_variable_list = []

    for model_name, feature_table in feature_tables.items():
        _, all_by_variable, _, mid_by_variable = aggregate_metrics_from_feature_table(
            feature_table=feature_table,
            model_name=model_name
        )
        all_by_variable_list.append(all_by_variable)
        mid_by_variable_list.append(mid_by_variable)

    table2_all_by_variable = pd.concat(all_by_variable_list, axis=0, ignore_index=True)
    table4_mid_by_variable = pd.concat(mid_by_variable_list, axis=0, ignore_index=True)

    table1_all_overall = (
        table2_all_by_variable
        .groupby("Model", as_index=False)[metric_cols]
        .mean()
    )

    table3_mid_overall = (
        table4_mid_by_variable
        .groupby("Model", as_index=False)[metric_cols]
        .mean()
    )

    out_xlsx = Path(out_xlsx)
    out_xlsx.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        table1_all_overall.to_excel(writer, sheet_name="table1_all_overall", index=False)
        table2_all_by_variable.to_excel(writer, sheet_name="table2_all_by_variable", index=False)
        table3_mid_overall.to_excel(writer, sheet_name="table3_mid_overall", index=False)
        table4_mid_by_variable.to_excel(writer, sheet_name="table4_mid_by_variable", index=False)

    prefix = out_xlsx.with_suffix("")
    table1_all_overall.to_csv(str(prefix) + "_table1_all_overall.csv", index=False, encoding="utf-8-sig")
    table2_all_by_variable.to_csv(str(prefix) + "_table2_all_by_variable.csv", index=False, encoding="utf-8-sig")
    table3_mid_overall.to_csv(str(prefix) + "_table3_mid_overall.csv", index=False, encoding="utf-8-sig")
    table4_mid_by_variable.to_csv(str(prefix) + "_table4_mid_by_variable.csv", index=False, encoding="utf-8-sig")

    print(f"\n指标表已保存: {out_xlsx}")

    print("\nTable 1: 全部时次总体指标，5变量平均")
    try:
        display(table1_all_overall)
    except Exception:
        print(table1_all_overall)

    print("\nTable 2: 全部时次分变量指标")
    try:
        display(table2_all_by_variable)
    except Exception:
        print(table2_all_by_variable)

    print("\nTable 3: 中间时次总体指标，不含 feature index = 0,4,5,9,10,14,15,19,20,24")
    try:
        display(table3_mid_overall)
    except Exception:
        print(table3_mid_overall)

    print("\nTable 4: 中间时次分变量指标")
    try:
        display(table4_mid_by_variable)
    except Exception:
        print(table4_mid_by_variable)

    return table1_all_overall, table2_all_by_variable, table3_mid_overall, table4_mid_by_variable


# ============================================================
# 5. 保存 nc 函数：每个变量保存为 data_slp/data_z300/.../data_v10
# ============================================================
def save_multivar_to_nc(arr, times, lat, lon, out_path, description, time_chunk=64, complevel=4):
    """
    arr shape: (N, H, W, 25)
    保存为：
        data_slp(time, feature, latitude, longitude)
        data_z300(time, feature, latitude, longitude)
        data_z500(time, feature, latitude, longitude)
        data_u10(time, feature, latitude, longitude)
        data_v10(time, feature, latitude, longitude)
    """

    from netCDF4 import Dataset, date2num

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    n, h, w, f = arr.shape
    assert f == 25

    pd_times = pd.to_datetime(times)
    py_times = [t.to_pydatetime() for t in pd_times]

    time_units = f"hours since {py_times[0].strftime('%Y-%m-%d %H:%M:%S')}"
    calendar = "standard"
    time_values = date2num(py_times, units=time_units, calendar=calendar)

    ds = Dataset(out_path, "w", format="NETCDF4")

    ds.createDimension("time", None)
    ds.createDimension("feature", 5)
    ds.createDimension("latitude", h)
    ds.createDimension("longitude", w)

    tvar = ds.createVariable("time", "f8", ("time",))
    fvar = ds.createVariable("feature", "i4", ("feature",))
    hvar = ds.createVariable("feature_hour", "i4", ("feature",))
    latvar = ds.createVariable("latitude", "f4", ("latitude",))
    lonvar = ds.createVariable("longitude", "f4", ("longitude",))

    tvar[:] = time_values
    tvar.units = time_units
    tvar.calendar = calendar

    fvar[:] = np.arange(5, dtype=np.int32)
    hvar[:] = FEATURE_HOURS
    latvar[:] = lat.astype(np.float32)
    lonvar[:] = lon.astype(np.float32)

    ds.description = description
    ds.variable_order = ",".join(VAR_NAMES)
    ds.feature_description = "feature 0-4 corresponds to +0,+6,+12,+18,+24 hours for each variable"
    ds.spatial_resolution = "0.5 degree"
    ds.data_status = "standardized"

    encoding_kwargs = {
        "zlib": True,
        "complevel": complevel,
        "chunksizes": (min(time_chunk, n), 5, h, w),
    }

    for vi, var_name in enumerate(VAR_NAMES):
        y0 = vi * 5
        nc_var_name = f"data_{var_name}"

        vout = ds.createVariable(
            nc_var_name,
            "f4",
            ("time", "feature", "latitude", "longitude"),
            **encoding_kwargs
        )

        for t0 in tqdm(range(0, n, time_chunk), desc=f"保存 {nc_var_name}"):
            t1 = min(n, t0 + time_chunk)
            block = np.asarray(arr[t0:t1, :, :, y0:y0+5], dtype=np.float32).transpose(0, 3, 1, 2)
            vout[t0:t1, :, :, :] = block
            del block
            gc.collect()

    ds.close()
    print(f"nc 已保存: {out_path}")


# ============================================================
# 6. baseline 需要的空间插值和光流函数
# ============================================================
def upsample_2x_like_xarray_no_extrap(x):
    """
    x shape: (N, h, w)
    输出 shape: (N, 2h, 2w)

    复现旧 xarray interp 的核心效果：
    - 偶数位置为原值
    - 中间位置线性插值
    - 最后一行/列因为没有外推，保留 NaN
    """

    x = np.asarray(x, dtype=np.float32)
    n, h, w = x.shape

    tmp = np.full((n, 2 * h, w), np.nan, dtype=np.float32)
    tmp[:, 0::2, :] = x
    tmp[:, 1:-1:2, :] = 0.5 * (x[:, :-1, :] + x[:, 1:, :])

    out = np.full((n, 2 * h, 2 * w), np.nan, dtype=np.float32)
    out[:, :, 0::2] = tmp
    out[:, :, 1:-1:2] = 0.5 * (tmp[:, :, :-1] + tmp[:, :, 1:])

    return out


def fill_edge_nan_for_cv2(x):
    """
    光流不能稳定处理 NaN。
    插值造成的最后一行/列 NaN 用邻近边界填充。
    仅用于光流计算，不改变 testy。
    """

    y = np.array(x, dtype=np.float32, copy=True)

    if y.shape[1] >= 2:
        y[:, -1, :] = y[:, -2, :]
    if y.shape[2] >= 2:
        y[:, :, -1] = y[:, :, -2]

    np.nan_to_num(y, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    return y


def optical_flow_half(frame0, frame1, desc="optical flow"):
    """
    frame0/frame1 shape: (N, H, W)
    返回从 frame0 到 frame1 的半步光流外推结果。
    """

    frame0 = np.asarray(frame0, dtype=np.float32)
    frame1 = np.asarray(frame1, dtype=np.float32)

    assert frame0.shape == frame1.shape

    n, h, w = frame0.shape
    out = np.empty((n, h, w), dtype=np.float32)

    grid_x, grid_y = np.meshgrid(
        np.arange(w, dtype=np.float32),
        np.arange(h, dtype=np.float32)
    )

    for i in tqdm(range(n), desc=desc):
        f0 = np.ascontiguousarray(frame0[i])
        f1 = np.ascontiguousarray(frame1[i])

        flow = cv2.calcOpticalFlowFarneback(
            f0,
            f1,
            None,
            0.5,
            3,
            15,
            3,
            5,
            1.1,
            0
        )

        map_x = grid_x + flow[..., 0] * 0.5
        map_y = grid_y + flow[..., 1] * 0.5

        out[i] = cv2.remap(
            f0,
            map_x.astype(np.float32),
            map_y.astype(np.float32),
            interpolation=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REPLICATE
        )

    return out


========== 读取并处理变量: slp ==========
[slp] nc变量名: msl
[slp] raw shape: (51136, 233, 377)
[slp] time: 1980-01-01 00:00:00 -> 2014-12-31 18:00:00
[slp] lat size: 233 lon size: 377
[slp] HR before norm: (51132, 116, 188, 5)
[slp] LR before norm: (51132, 58, 94, 2)
[slp] 写入 testx 通道 0:2
[slp] 写入 testy 通道 0:5

========== 读取并处理变量: z300 ==========
[z300] nc变量名: z
[z300] raw shape: (51136, 233, 377)
[z300] time: 1980-01-01 00:00:00 -> 2014-12-31 18:00:00
[z300] lat size: 233 lon size: 377
[z300] HR before norm: (51132, 116, 188, 5)
[z300] LR before norm: (51132, 58, 94, 2)
[z300] 写入 testx 通道 2:4
[z300] 写入 testy 通道 5:10

========== 读取并处理变量: z500 ==========
[z500] nc变量名: z
[z500] raw shape: (51136, 233, 377)
[z500] time: 1980-01-01 00:00:00 -> 2014-12-31 18:00:00
[z500] lat size: 233 lon size: 377
[z500] HR before norm: (51132, 116, 188, 5)
[z500] LR before norm: (51132, 58, 94, 2)
[z500] 写入 testx 通道 4:6
[z500] 写入 testy 通道 10:15

========== 读取并处理变量: u10 ==========
[u10] nc变量名: u10
[u10] raw shape

In [2]:
# ============================================================
# TS 代码框 2：
# baseline-TS = Temporal first + Spatial second
# 光流时间插值 -> 双线性空间插值
# ============================================================

TS_PRED_TMP_DIR = RESULT_DIR / "_tmp_baseline_TS_5vars_npy"
TS_PRED_TMP_DIR.mkdir(parents=True, exist_ok=True)

TS_PRED_NPY = TS_PRED_TMP_DIR / "predicty_baseline_TS_data.npy"

if TS_PRED_NPY.exists():
    TS_PRED_NPY.unlink()

pred_ts = open_memmap(
    TS_PRED_NPY,
    mode="w+",
    dtype=np.float32,
    shape=testy.shape
)

print("testx:", testx.shape)
print("testy:", testy.shape)
print("pred_ts:", pred_ts.shape)

for vi, var_name in enumerate(VAR_NAMES):
    print(f"\n========== baseline-TS 处理变量: {var_name} ==========")

    x0 = vi * 2
    y0 = vi * 5

    lr_t0 = np.asarray(testx[..., x0], dtype=np.float32)
    lr_t4 = np.asarray(testx[..., x0 + 1], dtype=np.float32)

    # 1. 先在 LR 网格做时间光流插值
    lr_t2 = optical_flow_half(
        lr_t0,
        lr_t4,
        desc=f"TS {var_name}: LR t0 -> t4, get t2"
    )

    lr_t1 = optical_flow_half(
        lr_t0,
        lr_t2,
        desc=f"TS {var_name}: LR t0 -> t2, get t1"
    )

    lr_t3 = optical_flow_half(
        lr_t2,
        lr_t4,
        desc=f"TS {var_name}: LR t2 -> t4, get t3"
    )

    # 2. 再对每个时间片做空间插值到 HR
    hr_t0 = upsample_2x_like_xarray_no_extrap(lr_t0)
    hr_t1 = upsample_2x_like_xarray_no_extrap(lr_t1)
    hr_t2 = upsample_2x_like_xarray_no_extrap(lr_t2)
    hr_t3 = upsample_2x_like_xarray_no_extrap(lr_t3)
    hr_t4 = upsample_2x_like_xarray_no_extrap(lr_t4)

    pred_ts[..., y0 + 0] = hr_t0
    pred_ts[..., y0 + 1] = hr_t1
    pred_ts[..., y0 + 2] = hr_t2
    pred_ts[..., y0 + 3] = hr_t3
    pred_ts[..., y0 + 4] = hr_t4

    pred_ts.flush()

    print(f"{var_name} 写入 pred_ts 通道 {y0}:{y0+5}")

    del lr_t0, lr_t4, lr_t1, lr_t2, lr_t3
    del hr_t0, hr_t1, hr_t2, hr_t3, hr_t4
    gc.collect()

pred_ts.flush()
del pred_ts
gc.collect()

print("\nbaseline-TS 降尺度完成，结果保存为:", TS_PRED_NPY)

testx: (10227, 58, 94, 10)
testy: (10227, 116, 188, 25)
pred_ts: (10227, 116, 188, 25)

========== baseline-TS 处理变量: slp ==========


TS slp: LR t2 -> t4, get t3: 100%|█████████████████████████████████████████████| 10227/10227 [00:09<00:00, 1045.12it/s]


slp 写入 pred_ts 通道 0:5

========== baseline-TS 处理变量: z300 ==========


TS z300: LR t2 -> t4, get t3: 100%|████████████████████████████████████████████| 10227/10227 [00:09<00:00, 1056.22it/s]


z300 写入 pred_ts 通道 5:10

========== baseline-TS 处理变量: z500 ==========


...

z500 写入 pred_ts 通道 10:15

========== baseline-TS 处理变量: u10 ==========


TS u10: LR t2 -> t4, get t3: 100%|█████████████████████████████████████████████| 10227/10227 [00:09<00:00, 1028.38it/s]


u10 写入 pred_ts 通道 15:20

========== baseline-TS 处理变量: v10 ==========


TS v10: LR t2 -> t4, get t3: 100%|█████████████████████████████████████████████| 10227/10227 [00:09<00:00, 1081.17it/s]


v10 写入 pred_ts 通道 20:25

baseline-TS 降尺度完成，结果保存为: E:\Dr_Research\result\_tmp_baseline_TS_5vars_npy\predicty_baseline_TS_data.npy


In [3]:
# ============================================================
# TS 代码框 3：计算 baseline-TS 指标
# ============================================================

pred_ts = np.load(TS_PRED_NPY, mmap_mode="r")

print("pred_ts:", pred_ts.shape)
print("testy  :", testy.shape)

if pred_ts.shape != testy.shape:
    raise ValueError(
        f"baseline-TS shape 不一致：pred_ts={pred_ts.shape}, testy={testy.shape}"
    )

feature_table_ts = calc_feature_metrics_table(
    y_true=testy,
    y_pred=pred_ts,
    hr_range_crop=metadata["hr_range_crop"],
    model_name="baseline_TS",
    ssim_batch_size=16
)

feature_table_ts_path = RESULT_DIR / "feature_metrics_baseline_TS_data.csv"
feature_table_ts.to_csv(feature_table_ts_path, index=False, encoding="utf-8-sig")
print("feature 级指标已保存:", feature_table_ts_path)

feature_tables_ts = {
    "baseline_TS": feature_table_ts
}

ts_metrics_xlsx = RESULT_DIR / "downscaling_5vars_baseline_TS_data_metrics_tables_MSE.xlsx"

table1_all_overall_ts, table2_all_by_variable_ts, table3_mid_overall_ts, table4_mid_by_variable_ts = save_metrics_tables(
    feature_tables=feature_tables_ts,
    out_xlsx=ts_metrics_xlsx
)

del pred_ts
gc.collect()

pred_ts: (10227, 116, 188, 25)
testy  : (10227, 116, 188, 25)


Feature metrics: baseline_TS: 100%|█████████████████████████████████████████████████| 25/25 [1:56:32<00:00, 279.71s/it]


feature 级指标已保存: E:\Dr_Research\result\feature_metrics_baseline_TS_data.csv

指标表已保存: E:\Dr_Research\result\downscaling_5vars_baseline_TS_data_metrics_tables_MSE.xlsx

Table 1: 全部时次总体指标，5变量平均


,Model,R,MSE,SSIM,PSNR
0,baseline_TS,0.85129,0.283334,0.786969,42.37145



Table 2: 全部时次分变量指标


...


Table 3: 中间时次总体指标，不含 feature index = 0,4,5,9,10,14,15,19,20,24


,Model,R,MSE,SSIM,PSNR
0,baseline_TS,0.75886,0.4599,0.653748,33.813793



Table 4: 中间时次分变量指标


,Model,Variable,R,MSE,SSIM,PSNR
0,baseline_TS,slp,0.762626,0.438857,0.663202,35.884823
1,baseline_TS,z300,0.885895,0.230683,0.743803,33.737641
2,baseline_TS,z500,0.828093,0.343883,0.710788,35.102840
3,baseline_TS,u10,0.690404,0.578143,0.609470,32.682812
4,baseline_TS,v10,0.627282,0.707936,0.541475,31.660848


570

In [4]:
# ============================================================
# TS 代码框 4：保存 baseline-TS predicty
# ============================================================
# 文件内部变量：
# data_slp, data_z300, data_z500, data_u10, data_v10
# ============================================================

pred_ts = np.load(TS_PRED_NPY, mmap_mode="r")

TS_NC_PATH = RESULT_DIR / "TS_model_data_standardized.nc"

save_multivar_to_nc(
    arr=pred_ts,
    times=metadata["test_times"],
    lat=metadata["lat"],
    lon=metadata["lon"],
    out_path=TS_NC_PATH,
    description="Standardized 5-variable baseline-TS result. Optical-flow temporal interpolation first, spatial interpolation second.",
    time_chunk=64,
    complevel=4
)

del pred_ts
gc.collect()

print("baseline-TS nc 保存完成:", TS_NC_PATH)

保存 data_v10: 100%|█████████████████████████████████████████████████████████████████| 160/160 [02:04<00:00,  1.29it/s]


nc 已保存: E:\Dr_Research\result\TS_model_data_standardized.nc
baseline-TS nc 保存完成: E:\Dr_Research\result\TS_model_data_standardized.nc
